# Diabetes Risk Prediction & Recommendation

This notebook loads a trained XGBoost model, predicts diabetes risk from health metrics,
explains top contributing factors using SHAP, and generates personalized health advice.

## 1. Install Dependencies
Run this cell once to install required packages into the virtual environment.

In [8]:
!pip install xgboost shap numpy scikit-learn langchain langchain-openai


[notice] A new release of pip is available: 25.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


## 2. Load Model

In [9]:
import pickle
import numpy as np
import shap

model = pickle.load(open("./model/xgb_model.pkl", "rb"))
print("Model loaded successfully.")

Model loaded successfully.


## 3. Helper Functions

In [10]:
FEATURE_NAMES = [
    "Pregnancies", "Glucose", "BloodPressure",
    "SkinThickness", "Insulin", "BMI",
    "DiabetesPedigreeFunction", "Age",
    "Missing_Insulin", "Missing_SkinThickness"
]


def prepare_input(data):
    """Take 8 raw features and return 10 features (with missing-value flags)."""
    (Pregnancies, Glucose, BloodPressure,
     SkinThickness, Insulin, BMI,
     DiabetesPedigreeFunction, Age) = data

    Missing_Insulin = 1 if Insulin == 0 else 0
    Missing_SkinThickness = 1 if SkinThickness == 0 else 0

    return [
        Pregnancies, Glucose, BloodPressure,
        SkinThickness, Insulin, BMI,
        DiabetesPedigreeFunction, Age,
        Missing_Insulin, Missing_SkinThickness
    ]


def predict_risk(data):
    """Predict diabetes probability and risk level from 10 prepared features."""
    data = np.array(data).reshape(1, -1)
    proba = model.predict_proba(data)[0][1]

    if proba < 0.3:
        risk = "Low"
    elif proba < 0.7:
        risk = "Medium"
    else:
        risk = "High"

    return proba, risk


def get_top_features(data, feature_names):
    """Use SHAP to find the top 3 features contributing to the prediction."""
    data = np.array(data).reshape(1, -1)

    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(data)[0]

    feature_impact = list(zip(feature_names, shap_values))
    feature_impact.sort(key=lambda x: abs(x[1]), reverse=True)

    result = []
    for f, v in feature_impact[:3]:
        impact = "High" if abs(v) > 0.3 else "Medium"
        result.append({"feature": f, "impact": impact})

    return result

print("Helper functions defined.")

Helper functions defined.


## 4. LLM Advice Generation (DeepSeek API)

This section uses the **DeepSeek API** to generate personalized health advice.

In [11]:
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="deepseek-chat",
    api_key="sk-90f7995e2bf14c5e8cdb3b6bdc0b1ed6",
    base_url="https://api.deepseek.com",
)
print("DeepSeek LLM loaded.")

prompt_template = PromptTemplate(
    input_variables=["risk", "probability", "factors"],
    template="""
You are a health assistant.

Risk Level: {risk}
Probability: {probability}

Top contributing factors:
{factors}

Provide:
1. Simple explanation
2. Lifestyle advice
3. Preventive steps
4. Suggest consulting a doctor

Do NOT give medical diagnosis.
""",
)


def generate_advice(risk, probability, factors):
    """Generate health advice using DeepSeek LLM."""
    formatted_factors = "\n".join(
        [f"- {f['feature']}: {f['impact']}" for f in factors]
    )
    final_prompt = prompt_template.format(
        risk=risk,
        probability=round(probability, 2),
        factors=formatted_factors,
    )
    response = llm.invoke(final_prompt)
    return response.content

DeepSeek LLM loaded.


## 5. Full Analysis Pipeline

In [12]:
def analyze_risk(input_data):
    """
    Run the full pipeline: prepare input -> predict -> explain -> advise.

    Parameters
    ----------
    input_data : list of 8 numbers
        [Pregnancies, Glucose, BloodPressure, SkinThickness,
         Insulin, BMI, DiabetesPedigreeFunction, Age]

    Returns
    -------
    dict with keys: risk, probability, top_factors, advice
    """
    full_input = prepare_input(input_data)
    probability, risk = predict_risk(full_input)
    factors = get_top_features(full_input, FEATURE_NAMES)
    advice = generate_advice(risk, probability, factors)

    return {
        "risk": risk,
        "probability": round(probability, 4),
        "top_factors": factors,
        "advice": advice,
    }

print("analyze_risk() ready.")

analyze_risk() ready.


In [13]:
# Example: [Pregnancies, Glucose, BP, SkinThickness, Insulin, BMI, DPF, Age]
sample_input = [2, 180, 85, 30, 0, 32.5, 0.5, 45]

result = analyze_risk(sample_input)

print(f"Risk Level : {result['risk']}")
print(f"Probability: {result['probability']}")
print(f"Top Factors: {result['top_factors']}")
print()
print("--- Advice ---")
print(result['advice'])

Risk Level : High
Probability: 0.7171000242233276
Top Factors: [{'feature': 'Glucose', 'impact': 'High'}, {'feature': 'Insulin', 'impact': 'High'}, {'feature': 'Age', 'impact': 'High'}]

--- Advice ---
**1. Simple Explanation**
Based on the assessment, there is a significant indication of elevated health risk. The primary factors contributing to this are high levels of glucose (sugar) and insulin in your body, combined with your age. High glucose and insulin often work together and can indicate that your body is struggling to manage blood sugar effectively, which is a major concern for long-term health.

**2. Lifestyle Advice**
*   **Diet:** Focus on a whole-food, low-glycemic diet. Reduce intake of sugary drinks, refined carbohydrates (white bread, pasta, pastries), and processed foods. Increase fiber from vegetables, legumes, and whole grains. Prioritize lean proteins and healthy fats.
*   **Exercise:** Engage in regular physical activity. Aim for at least 150 minutes of moderate-int

## 6. Run Prediction

Edit the values below to test with different patient data:

| Feature | Description | Typical Range |
|---------|-------------|---------------|
| Pregnancies | Number of pregnancies | 0–15 |
| Glucose | Blood glucose (mg/dL) | 70–200 |
| BloodPressure | Diastolic BP (mmHg) | 40–120 |
| SkinThickness | Triceps fold (mm) | 0–60 |
| Insulin | 2-hr serum insulin (mu U/ml) | 0–300 |
| BMI | Body Mass Index | 15–45 |
| DiabetesPedigreeFunction | Family history score | 0.0–2.5 |
| Age | Age in years | 20–70 |